# ALS: влияние параметров модели

После опытов 04–06 фиксируем **score_coef=0, planned=0,5 и фильтр 5/5**.
Остальные веса статусов прежние, оценки <5 исключены, затухание выключено.

Проверяем исходную конфигурацию и по одному меняем:
- factors: 64 → 32 / 128;
- regularization: 0,1 → 0,01 / 1;
- alpha: 20 → 5 / 80;
- iterations: 20 → 40.

Восемь конфигураций × seed 42/7/2026 = **24 запуска**. num_threads=4,
confidence=1+alpha*weight; внутренняя alpha библиотеки остаётся 1.
Это ограниченное однофакторное сравнение, а не поиск глобального оптимума:
сочетания изменений и их взаимодействия здесь не исследуются.

Все варианты обучаем по одним положительным парам. Фильтрация только train;
полная train-history для исключений. Сохраняем всех 7 184 warm-validation
пользователей и все цели независимо от доступности модели; fallback отсутствует.
Основной критерий — средний NDCG@10; дополнительно разброс seed, парные разницы,
precision/recall, покрытие и время. Test не оцениваем. Последний период уже изучался
в старом EDA, его независимость ограничена. Defaults и исходную модель не меняем.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import sys
import json
from pathlib import Path
from time import perf_counter
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))
from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate, ndcg_at_k
from app.training.weights import weighted_interactions, DEFAULT_STATUS_WEIGHTS
from app.recsys.popularity import PopularityRecommender
from app.recsys.als import ALSRecommender

pd.set_option("display.float_format", "{:,.6f}".format)
VALIDATION_START = pd.Timestamp("2026-09-02T00:34:23Z")
TEST_START = pd.Timestamp("2026-09-30T00:34:23Z")
SNAPSHOT_END = pd.Timestamp("2026-10-28T00:34:23Z")
SCORE_COEF = 0.0
BASE_STATUS_WEIGHTS = DEFAULT_STATUS_WEIGHTS.copy()
assert BASE_STATUS_WEIGHTS == dict(completed=1.0, watching=1.0, on_hold=0.5, planned=0.5, dropped=0.0)
SEEDS = [42, 7, 2026]
ALS_PARAMS = dict(factors=64, regularization=0.1, alpha=20, iterations=20, num_threads=4)
versions = {name: importlib.metadata.version(name) for name in ["implicit", "numpy", "scipy", "pandas"]}
print(versions)
inter = load_anilist()
assert inter.updated_at.max() == SNAPSHOT_END, "Снимок изменился: пересмотрите протокол"
history, validation, _ = chronological_split(
    inter, validation_start=VALIDATION_START, test_start=TEST_START)
training = filter_interactions(history, min_user_items=5, min_item_users=5)
counts = history.user_id.value_counts()
targets = {u: items for u, items in relevant_items(validation).items() if counts.get(u, 0) >= 5}
users = sorted(targets)
seen = history.groupby("user_id").anime_id.agg(set).to_dict()
print(f"Train после фильтра: {len(training):,} записей; warm-validation: {len(users):,} пользователей")
assert len(users) == 7184
assert len(history) == 3513393 and len(training) == 3499908
assert len(validation) == 218925
# Однофакторное сравнение: варианты фиксируем до просмотра результатов.
CONFIGS = {
    "baseline": ALS_PARAMS.copy(),
    "factors_32": {**ALS_PARAMS, "factors": 32},
    "factors_128": {**ALS_PARAMS, "factors": 128},
    "regularization_0.01": {**ALS_PARAMS, "regularization": 0.01},
    "regularization_1": {**ALS_PARAMS, "regularization": 1.0},
    "alpha_5": {**ALS_PARAMS, "alpha": 5},
    "alpha_80": {**ALS_PARAMS, "alpha": 80},
    "iterations_40": {**ALS_PARAMS, "iterations": 40},
}
weighted = weighted_interactions(training, status_weights=BASE_STATUS_WEIGHTS.copy(),
                                 score_coef=SCORE_COEF, min_score=5, half_life_days=None)
assert len(weighted) == 3293589
assert not weighted.status.eq("dropped").any()
assert not weighted.score.lt(5).fillna(False).any()
assert DEFAULT_STATUS_WEIGHTS == BASE_STATUS_WEIGHTS
out = Path.cwd().parent / "artifacts" / "als_hyperparameters"
out.mkdir(parents=True, exist_ok=True)


/home/deonik3/AniFolder/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'implicit': '0.7.3', 'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0'}


Train после фильтра: 3,499,908 записей; warm-validation: 7,184 пользователей


## Контрольная популярность

Прежний baseline за 28 дней на train 5/5: NDCG@10 ≈ 0,113021.

In [2]:
popular = PopularityRecommender(window_days=28).fit(training, now=VALIDATION_START, history=history)
popular_metrics = evaluate(popular.recommend_many(users, k=10), targets)
assert np.isclose(popular_metrics["ndcg@10"], 0.113021, atol=5e-7, rtol=0)
pd.Series(popular_metrics, name="популярность 28 дней").to_frame()

,популярность 28 дней
precision@10,0.099624
recall@10,0.064522
ndcg@10,0.113021
coverage@10,65.000000
users,"7,184.000000"


## 24 запуска

Исходные данные и история исключений одинаковы во всех запусках.

In [3]:
records = []
per_user = {}
(out / "progress.log").write_text("")
for name, params in CONFIGS.items():
    for seed in SEEDS:
        started = perf_counter()
        model = ALSRecommender(**params, random_state=seed).fit(weighted, history=history)
        fit_seconds = perf_counter() - started
        started = perf_counter()
        recs = model.recommend_many(users, k=10)
        recommend_seconds = perf_counter() - started
        assert set(recs) == set(users)
        assert all(len(r) == len(set(r)) and not (set(r) & seen.get(u, set())) for u, r in recs.items())
        metrics = evaluate(recs, targets)
        per_user[(name, seed)] = np.array([ndcg_at_k(recs[u], targets[u], 10) for u in users])
        records.append({"configuration": name, "seed": seed, **params, **metrics,
                        "training_rows": len(weighted),
                        "short_lists": sum(len(r) < 10 for r in recs.values()),
                        "fit_seconds": fit_seconds, "recommend_seconds": recommend_seconds})
        message = (f"{len(records)}/{len(CONFIGS)*len(SEEDS)}: {name}, seed={seed}: "
                   f"NDCG@10={metrics['ndcg@10']:.6f}, coverage={metrics['coverage@10']}, "
                   f"train={fit_seconds:.1f}s")
        print(message, flush=True)
        with (out / "progress.log").open("a") as log:
            log.write(message + "\n")
results = pd.DataFrame(records)
display(results.set_index(["configuration", "seed"]))

1/24: baseline, seed=42: NDCG@10=0.100216, coverage=4253, train=7.1s


2/24: baseline, seed=7: NDCG@10=0.099340, coverage=4219, train=6.9s


3/24: baseline, seed=2026: NDCG@10=0.099730, coverage=4256, train=6.8s


4/24: factors_32, seed=42: NDCG@10=0.104307, coverage=4053, train=5.8s


5/24: factors_32, seed=7: NDCG@10=0.105819, coverage=4033, train=5.7s


6/24: factors_32, seed=2026: NDCG@10=0.105097, coverage=3987, train=5.6s


... (15 outputs left out)

22/24: iterations_40, seed=42: NDCG@10=0.099531, coverage=4213, train=12.2s


23/24: iterations_40, seed=7: NDCG@10=0.099166, coverage=4233, train=11.6s


24/24: iterations_40, seed=2026: NDCG@10=0.098898, coverage=4253, train=12.2s


factors  regularization  alpha  iterations  \
configuration       seed                                               
baseline            42         64        0.100000     20          20   
                    7          64        0.100000     20          20   
                    2026       64        0.100000     20          20   
factors_32          42         32        0.100000     20          20   
                    7          32        0.100000     20          20   
                    2026       32        0.100000     20          20   
factors_128         42        128        0.100000     20          20   
                    7         128        0.100000     20          20   
                    2026      128        0.100000     20          20   
regularization_0.01 42         64        0.010000     20          20   
                    7          64        0.010000     20          20   
                    2026       64        0.010000     20          20   
regularization_1    42         64        1.000000     20          20   
                    7          64        1.000000     20          20   
                    2026       64        1.000000     20          20   
alpha_5             42         64        0.100000      5          20   
                    7          64        0.100000      5          20   
                    2026       64        0.100000      5          20   
alpha_80            42         64        0.100000     80          20   
                    7          64        0.100000     80          20   
                    2026       64        0.100000     80          20   
iterations_40       42         64        0.100000     20          40   
                    7          64        0.100000     20          40   
                    2026       64        0.100000     20          40   

                          num_threads  precision@10  recall@10  ndcg@10  \
configuration       seed                                                  
baseline            42              4      0.081974   0.064405 0.100216   
                    7               4      0.081584   0.063838 0.099340   
                    2026            4      0.081807   0.064172 0.099730   
factors_32          42              4      0.085732   0.065769 0.104307   
                    7               4      0.086498   0.067369 0.105819   
                    2026            4      0.086414   0.066436 0.105097   
factors_128         42              4      0.079079   0.060676 0.096890   
                    7               4      0.080178   0.061484 0.097640   
                    2026            4      0.079594   0.061962 0.097708   
regularization_0.01 42              4      0.081278   0.063530 0.099456   
                    7               4      0.081180   0.063446 0.099044   
                    2026            4      0.080818   0.063124 0.098390   
regularization_1    42              4      0.082739   0.065248 0.100955   
                    7               4      0.081974   0.064367 0.099785   
                    2026            4      0.081751   0.063194 0.099628   
alpha_5             42              4      0.089797   0.066203 0.108484   
                    7               4      0.088906   0.065011 0.106986   
                    2026            4      0.088850   0.064417 0.107470   
alpha_80            42              4      0.065159   0.055127 0.080248   
                    7               4      0.065883   0.055071 0.081174   
                    2026            4      0.066857   0.056354 0.082649   
iterations_40       42              4      0.081055   0.063206 0.099531   
                    7               4      0.080972   0.063685 0.099166   
                    2026            4      0.081194   0.063427 0.098898   

                          coverage@10  users  training_rows  short_lists  \
configuration       seed                                                   
baseline            42           4253   7184      

## Качество и чувствительность к seed

Std описывает инициализацию на одном validation. Подбор среди нескольких вариантов может завышать качество выбранного кандидата; три seed не заменяют независимую проверку.

In [4]:
summary = results.groupby("configuration", sort=False).agg(
    ndcg_mean=("ndcg@10", "mean"), ndcg_std=("ndcg@10", "std"),
    ndcg_min=("ndcg@10", "min"), ndcg_max=("ndcg@10", "max"),
    precision_mean=("precision@10", "mean"), recall_mean=("recall@10", "mean"),
    coverage_mean=("coverage@10", "mean"), fit_seconds_mean=("fit_seconds", "mean"),
    recommend_seconds_mean=("recommend_seconds", "mean"))
display(summary.sort_values("ndcg_mean", ascending=False))
candidate = summary.ndcg_mean.idxmax()
print("Лучший вариант по среднему NDCG:", candidate, CONFIGS[candidate])
print("NDCG baseline популярности:", popular_metrics["ndcg@10"])
paired = results.pivot(index="seed", columns="configuration", values="ndcg@10")
paired_deltas = paired.subtract(paired["baseline"], axis=0)
display(paired_deltas)
delta_rows = []
for name in CONFIGS:
    if name == "baseline":
        continue
    delta = np.mean([per_user[(name, seed)] - per_user[("baseline", seed)] for seed in SEEDS], axis=0)
    delta_rows.append({"configuration": name, "mean_delta": delta.mean(),
                       "лучше": int((delta > 1e-12).sum()),
                       "без изменения": int((np.abs(delta) <= 1e-12).sum()),
                       "хуже": int((delta < -1e-12).sum())})
user_deltas = pd.DataFrame(delta_rows).set_index("configuration")
display(user_deltas)

,ndcg_mean,ndcg_std,ndcg_min,ndcg_max,precision_mean,recall_mean,coverage_mean,fit_seconds_mean,recommend_seconds_mean
configuration,,,,,,,,,
alpha_5,0.107646,0.000765,0.106986,0.108484,0.089184,0.065210,"3,105.000000",6.731741,9.395271
factors_32,0.105074,0.000756,0.104307,0.105819,0.086215,0.066525,"4,024.333333",5.671092,8.752836
regularization_1,0.100123,0.000725,0.099628,0.100955,0.082155,0.064270,"4,265.333333",7.125454,9.441686
baseline,0.099762,0.000438,0.099340,0.100216,0.081788,0.064138,"4,242.666667",6.921547,8.975178
iterations_40,0.099198,0.000318,0.098898,0.099531,0.081074,0.063439,"4,233.000000",11.980662,8.973153
regularization_0.01,0.098963,0.000537,0.098390,0.099456,0.081092,0.063367,"4,228.666667",6.775136,9.382559
factors_128,0.097413,0.000454,0.096890,0.097708,0.079617,0.061374,"4,037.666667",8.871909,10.181837
alpha_80,0.081357,0.001211,0.080248,0.082649,0.065966,0.055517,"5,607.333333",6.639438,9.132396


Лучший вариант по среднему NDCG: alpha_5 {'factors': 64, 'regularization': 0.1, 'alpha': 5, 'iterations': 20, 'num_threads': 4}
NDCG baseline популярности: 0.11302101779723082


configuration,alpha_5,alpha_80,baseline,factors_128,factors_32,iterations_40,regularization_0.01,regularization_1
seed,,,,,,,,
7,0.007645,-0.018166,0.000000,-0.001701,0.006479,-0.000175,-0.000297,0.000445
42,0.008268,-0.019968,0.000000,-0.003325,0.004092,-0.000685,-0.000760,0.000739
2026,0.007740,-0.017081,0.000000,-0.002022,0.005367,-0.000832,-0.001340,-0.000102


,mean_delta,лучше,без изменения,хуже
configuration,,,,
factors_32,0.005312,1853,3551,1780
factors_128,-0.002349,1998,3302,1884
regularization_0.01,-0.000799,1193,4525,1466
regularization_1,0.000361,1486,4353,1345
alpha_5,0.007884,1987,3546,1651
alpha_80,-0.018405,1496,3424,2264
iterations_40,-0.000564,1488,4128,1568


## Артефакты опыта

Результаты в `ml/artifacts/als_hyperparameters/` (вне Git) и выводах ноутбука. Исходные артефакты не перезаписываем.

In [5]:
results.to_csv(out / "runs.csv", index=False)
summary.to_csv(out / "summary.csv")
paired_deltas.to_csv(out / "paired_deltas.csv")
user_deltas.to_csv(out / "user_deltas.csv")
(out / "protocol.json").write_text(json.dumps({
    "snapshot_end": SNAPSHOT_END.isoformat(), "validation_start": VALIDATION_START.isoformat(),
    "test_start": TEST_START.isoformat(), "test_evaluated": False,
    "min_user_items": 5, "min_item_users": 5, "score_coef": SCORE_COEF,
    "status_weights": BASE_STATUS_WEIGHTS, "min_score": 5, "min_user_scores": 5,
    "clip": [0.5, 1.5], "half_life_days": None, "seeds": SEEDS,
    "configurations": CONFIGS, "confidence": "1 + alpha * weight",
    "search_design": "one parameter at a time", "selection_metric": "mean warm-validation ndcg@10",
    "candidate": candidate, "warm_validation_users": len(users),
    "target_filtering": False, "exclusion_history": "full train",
    "user_id_namespace": "anilist", "item_id_namespace": "anilist",
    "versions": versions, "popular_baseline": popular_metrics,
}, ensure_ascii=False, indent=2) + "\n")
print("Сохранены:", ", ".join(p.name for p in sorted(out.iterdir())))

Сохранены: paired_deltas.csv, progress.log, protocol.json, runs.csv, summary.csv, user_deltas.csv


## Вывод

| Изменение относительно исходной ALS | Средний NDCG@10 | Std между seed | Среднее coverage@10 |
|---|---:|---:|---:|
| baseline | 0.099762 | 0.000438 | 4242.7 |
| factors_32 | 0.105074 | 0.000756 | 4024.3 |
| factors_128 | 0.097413 | 0.000454 | 4037.7 |
| regularization_0.01 | 0.098963 | 0.000537 | 4228.7 |
| regularization_1 | 0.100123 | 0.000725 | 4265.3 |
| alpha_5 | 0.107646 | 0.000765 | 3105.0 |
| alpha_80 | 0.081357 | 0.001211 | 5607.3 |
| iterations_40 | 0.099198 | 0.000318 | 4233.0 |

- **Лучший проверенный вариант — alpha=5**, при factors=64, regularization=0,1
  и iterations=20. Средний NDCG увеличивается на 0,007884 (~7,90%);
  парный выигрыш есть на всех трёх seed: от +0,007645 до +0,008268.
  Средний precision=0,089184 и recall=0,065210 также выше исходных значений.
- Цена этого улучшения — снижение среднего покрытия с 4242,7 до 3105 тайтлов
  (~−26,8%). Coverage измеряет число разных рекомендованных тайтлов,
  а не качество персонализации само по себе.
- **Factors=32** также улучшает NDCG на каждом seed: среднее 0,105074
  (~+5,32%), recall=0,066525, coverage=4024,3. Это лучший средний recall
  среди проверенных конфигураций. Увеличение числа факторов до 128 ухудшает NDCG.
- Regularization=1 даёт небольшой средний выигрыш (~0,36%), но не на каждом
  seed. Regularization=0,01, alpha=80 и 40 итераций ухудшают NDCG при каждом seed.
  Больше итераций увеличивает среднее время обучения с ~6,9 до ~12,0 с;
  это локальные замеры, не нагрузочный тест сервиса.
- Популярность за 28 дней остаётся выше: **0,113021** против 0,107646 у лучшего
  ALS. Разрыв с ней сократился, но победы ALS в этом опыте нет.
- Все 24 запуска используют одни 3 293 589 положительных пар и возвращают
  полный топ-10 всем 7 184 warm-validation пользователям. Нет дублей и пересечений
  с полной train-history. Исходная конфигурация точно воспроизвела все метрики
  опыта 06 на каждом seed.
- Это предварительный выбор по одному validation: три seed показывают
  чувствительность к инициализации, а не независимую проверку качества.
  Подбор вариантов по этому же validation может завышать оценку кандидата.
- **Следующий короткий опыт:** сочетание factors=32 и alpha=5 против
  factors=64, alpha=5 (лучшего отдельного изменения), при regularization=0,1,
  iterations=20 и тех же трёх seed. Выигрыши отдельных изменений не обязательно
  складываются. После этого можно зафиксировать конфигурацию для переобучения
  на train+validation и финальной оценки с оговоркой о ранее изученном test.
  Defaults и исходные артефакты не менялись; test здесь не оценивался.
